In [12]:
import re
import os

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer
from sklearn.utils import Bunch


DATA_DIR = "./datasets/20newsgroups"
RANDOM_SEED = 8888
MIN_WORDS = 10

# Load dataset
data = fetch_20newsgroups(data_home=DATA_DIR, subset="all", random_state=RANDOM_SEED)
data_train = fetch_20newsgroups(data_home=DATA_DIR, subset="train", random_state=RANDOM_SEED)
data_test = fetch_20newsgroups(data_home=DATA_DIR, subset="test", random_state=RANDOM_SEED)
assert isinstance(data, Bunch)

categories = data.target_names
print(f"Categories: {categories}")

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [13]:
# Preprocessing for the dataset
def strip_headers(text):
    """
    Remove headers from a raw newsgroup post.
    Headers end at the first blank line.
    """
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if line.strip() == "":
            return "\n".join(lines[i+1:])
    return text

def strip_footers(text):
    """
    Remove footers (signatures separated by '--' or long trailing blocks).
    """
    return re.split(r"\n--+\n", text)[0]

def strip_quotes(text):
    """
    Remove quoted lines starting with '>'.
    """
    return "\n".join([line for line in text.split("\n") if not line.strip().startswith(">")])

def clean_text(raw_text):
    """
    Apply all cleaning steps: headers, footers, quotes.
    """
    text = strip_headers(raw_text)
    text = strip_footers(text)
    text = strip_quotes(text)
    return text.strip()

In [14]:
def clean_split(raw_texts, raw_labels):
    """
    Clean, deduplicate, and drop too-short documents for one split (train or test).
    Kept separate per split so no information crosses from test into train.
    """
    # Clean texts
    texts, labels = [], []
    for raw_text, label in zip(raw_texts, raw_labels):
        cleaned = clean_text(raw_text)
        if cleaned:
            texts.append(cleaned)
            labels.append(label)

    # Deduplicate texts
    seen = set()
    unique_texts, unique_labels = [], []
    for text, label in zip(texts, labels):
        if text not in seen:
            seen.add(text)
            unique_texts.append(text)
            unique_labels.append(label)

    # Filter out too-short texts
    filtered_texts, filtered_labels = [], []
    for text, label in zip(unique_texts, unique_labels):
        if len(text.split()) > MIN_WORDS:
            filtered_texts.append(text)
            filtered_labels.append(label)

    return filtered_texts, filtered_labels

In [15]:
assert isinstance(data_train, Bunch)
assert isinstance(data_test, Bunch)
train_texts, y_train = clean_split(data_train.data, data_train.target)
test_texts, y_test = clean_split(data_test.data, data_test.target)

print(f"Train docs: {len(train_texts)} (raw: {len(data_train.data)})")
print(f"Test docs: {len(test_texts)} (raw: {len(data_test.data)})")

Train docs: 11099 (raw: 11314)
Test docs: 7377 (raw: 7532)


In [16]:
# Feature engineering: fit only on train, transform test to avoid leakage
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

# Feature scaling or normalization
normalizer = Normalizer()
X_train = normalizer.fit_transform(X_train)
X_test = normalizer.transform(X_test)

# Feature selection
selector = SelectKBest(chi2, k=5000)
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:20])
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Vocabulary size: 20000
Sample features: ['00' '000' '0001' '00010001b' '000152' '001' '00100010b' '0013' '0062'
 '007' '01' '01000100b' '0114' '0123456789' '02' '02106' '02139' '02238'
 '02p' '03']
X_train shape: (11099, 5000)
X_test shape: (7377, 5000)
